# Filtreleme ve Kategorizasyon

pipeline_v6.ipynb tarafindan uretilen ham dataseti alir, kategorilendirir ve filtreler.

**Akis:**
1. Ham dataset + enriched_repos yukle
2. Projeleri kategorilere ayir (AI/ML, Web, Data, vb.)
3. Placeholder filtreler uygula (proje + dosya seviyesi)
4. Filtreleme sonrasi ozet ve gorsellestirme
5. Filtrelenmis dataseti kaydet → model egitimi icin hazir

In [ ]:
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

---
## Bölüm 1: Veri Yükle

In [ ]:
WORK_DIR   = Path(r"C:\Users\bm123\Documents\MetricHunter")
OUTPUT_DIR = WORK_DIR / "output"

# En son uretilen ham dataset
full_files = sorted(OUTPUT_DIR.glob("dataset_full_*.csv"))
if not full_files:
    raise FileNotFoundError(f"{OUTPUT_DIR} icinde dataset_full_*.csv bulunamadi!")
DATA_PATH = full_files[-1]
print(f"Dataset: {DATA_PATH.name}")

df_raw = pd.read_csv(DATA_PATH)
print(f"Ham veri: {len(df_raw)} dosya, {df_raw['project'].nunique()} proje")

# enriched_repos.json — kategorizasyon icin topics + description
enriched_path = OUTPUT_DIR / "enriched_repos.json"
if enriched_path.exists():
    with open(enriched_path, "r", encoding="utf-8") as f:
        enriched_repos = json.load(f)
    repo_meta = {r["full_name"]: r for r in enriched_repos}
    print(f"Enriched repos: {len(repo_meta)} proje")
else:
    repo_meta = {}
    print("UYARI: enriched_repos.json bulunamadi — sadece dataset'teki bilgiler kullanilacak")

df = df_raw.copy()

In [ ]:
# Label'i yeniden hesapla — pd.qcut ile %50/%50 garanti
df["label"] = pd.qcut(df["commit_count"], q=2, labels=[0, 1], duplicates="drop").astype(int)
print(f"Commit median esigi: {df['commit_count'].median()}")
print(f"Label dengesi: 0={(df['label']==0).sum()} ({(df['label']==0).mean():.1%}), "
      f"1={(df['label']==1).sum()} ({(df['label']==1).mean():.1%})")

---
## Bölüm 2: Kategorizasyon

Her projeye GitHub topics + description keyword eslesmesiyle 1+ kategori atanir.

**Kategoriler:** AI/ML, Web, Data, DevOps/CLI, Security, Desktop, Mobile, Diger

In [ ]:
# Her kategori icin eslesecek keyword'ler (topics veya description'da aranir)
# Tire iceren keyword'ler hem 'machine-learning' hem 'machine learning' ile eslesir
CATEGORY_KEYWORDS = {
    "AI/ML": [
        # Genel AI/ML
        "machine-learning", "deep-learning", "artificial-intelligence", "neural-network",
        "neural-networks", "ai", "ml", "generative-ai", "foundation-model",
        # Alt alanlar
        "nlp", "natural-language-processing", "computer-vision", "reinforcement-learning",
        "reinforcement", "supervised-learning", "unsupervised-learning", "transfer-learning",
        "federated-learning", "self-supervised", "contrastive-learning",
        "time-series", "anomaly-detection", "object-detection", "image-segmentation",
        "image-classification", "text-classification", "sentiment-analysis",
        "named-entity-recognition", "question-answering", "summarization", "translation",
        "speech-recognition", "text-to-speech", "audio", "multimodal",
        # Frameworkler
        "tensorflow", "pytorch", "keras", "jax", "flax", "paddle", "mxnet",
        "scikit-learn", "sklearn", "xgboost", "lightgbm", "catboost",
        "huggingface", "transformers", "diffusers", "accelerate", "peft",
        "langchain", "llamaindex", "llama-index", "openai", "anthropic",
        # Modeller / mimariler
        "llm", "gpt", "bert", "gpt2", "gpt3", "gpt4", "llama", "mistral",
        "stable-diffusion", "diffusion", "vae", "gan", "autoencoder",
        "attention", "transformer", "encoder", "decoder",
        # Gorevler / uygulamalar
        "embedding", "rag", "retrieval", "vector-database", "vector-search",
        "chatbot", "chat", "recommendation", "recommender", "ranking",
        "ocr", "pose-estimation", "depth-estimation", "3d",
        # MLOps
        "mlops", "mlflow", "wandb", "experiment-tracking", "model-serving",
        "model-deployment", "inference", "onnx", "triton", "bentoml",
    ],
    "Web": [
        # Frameworkler
        "django", "flask", "fastapi", "aiohttp", "tornado", "starlette",
        "sanic", "bottle", "pyramid", "falcon", "litestar", "quart",
        "web-framework", "web-server", "wsgi", "asgi",
        # API
        "rest-api", "restful", "graphql", "grpc", "rpc", "api", "openapi",
        "swagger", "webhook", "microservice", "microservices",
        # Protokol / iletisim
        "http", "https", "websocket", "webrtc", "sse", "mqtt",
        # Frontend / template
        "html", "css", "javascript", "typescript", "react", "vue", "svelte",
        "jinja", "template", "frontend", "backend", "fullstack",
        # Scraping / crawling
        "scraping", "web-scraping", "crawler", "crawling", "spider",
        "beautifulsoup", "scrapy", "playwright", "selenium", "puppeteer",
        # Auth / guvenlik
        "oauth", "oauth2", "jwt", "saml", "sso",
        # HTTP istemcileri
        "requests", "httpx", "aiohttp", "urllib",
        # Genel web
        "web", "website", "webapp", "cms", "blog", "e-commerce", "ecommerce",
        "proxy", "reverse-proxy", "load-balancer", "cdn",
    ],
    "Data": [
        # Veri bilimi
        "data-science", "data-analysis", "data-analytics", "data-engineering",
        "data-pipeline", "data-processing", "data-wrangling", "data-cleaning",
        "exploratory-data-analysis", "eda", "statistics", "statistical",
        # Kutuphaneler
        "pandas", "numpy", "scipy", "polars", "dask", "vaex", "modin",
        # Big data
        "spark", "pyspark", "hadoop", "hive", "flink", "kafka", "kinesis",
        "big-data", "distributed", "streaming", "batch-processing",
        # Pipeline / orkestrasyon
        "etl", "elt", "airflow", "prefect", "dagster", "luigi", "dbt",
        "data-warehouse", "lakehouse", "delta-lake",
        # Veritabani
        "database", "sql", "postgresql", "postgres", "mysql", "sqlite",
        "mongodb", "redis", "elasticsearch", "cassandra", "clickhouse",
        "dynamodb", "neo4j", "graph-database", "time-series-database",
        "orm", "sqlalchemy", "alembic",
        # Gorsellestirme
        "visualization", "matplotlib", "seaborn", "plotly", "bokeh",
        "dash", "streamlit", "gradio", "panel", "altair",
        # Ortam
        "jupyter", "notebook", "colab", "jupyterlab",
    ],
    "DevOps/CLI": [
        # CLI
        "cli", "command-line", "command-line-tool", "terminal", "shell",
        "bash", "zsh", "fish", "tui", "curses", "rich", "typer", "click",
        "argparse", "tqdm",
        # Container / orkestrasyom
        "docker", "container", "kubernetes", "k8s", "helm", "podman",
        "docker-compose", "compose",
        # IaC
        "terraform", "ansible", "puppet", "chef", "saltstack",
        "infrastructure", "infrastructure-as-code", "iac",
        # CI/CD
        "ci-cd", "cicd", "github-actions", "gitlab-ci", "jenkins", "circleci",
        "continuous-integration", "continuous-deployment", "pipeline",
        # DevOps genel
        "devops", "sre", "platform-engineering", "gitops", "argocd",
        # Monitoring
        "monitoring", "observability", "logging", "tracing", "metrics",
        "prometheus", "grafana", "alerting", "opentelemetry",
        # Deploy / bulut
        "deploy", "deployment", "serverless", "lambda", "cloud",
        "aws", "gcp", "azure",
        # Genel araçlar
        "automation", "tool", "utility", "script", "helper",
        "package-manager", "build-tool", "linter", "formatter",
        "git", "version-control",
    ],
    "Security": [
        # Genel
        "security", "cybersecurity", "infosec", "appsec",
        # Kriptografi
        "cryptography", "crypto", "encryption", "decryption", "hashing",
        "tls", "ssl", "certificate", "pki",
        # Saldiri / savunma
        "vulnerability", "exploit", "cve", "poc",
        "penetration-testing", "pentest", "red-team", "blue-team",
        "ctf", "capture-the-flag", "wargame",
        # Malware / analiz
        "malware", "ransomware", "virus", "trojan",
        "forensics", "reverse-engineering", "disassembler", "decompiler",
        "binary-analysis", "fuzzing", "fuzzer",
        # Ag guvenligi
        "firewall", "ids", "ips", "network-security", "packet", "wireshark",
        "nmap", "scanner", "port-scanner",
        # Kimlik dogrulama
        "authentication", "authorization", "access-control", "rbac",
        "password", "secrets", "vault", "keystore",
        # Gizlilik
        "privacy", "anonymization", "gdpr", "tor", "vpn", "proxy",
        # SAST / DAST
        "sast", "dast", "code-analysis", "static-analysis", "audit",
    ],
    "Desktop": [
        # GUI frameworkleri
        "gui", "desktop", "desktop-app", "desktop-application",
        "tkinter", "pyqt", "pyqt5", "pyqt6", "pyside", "pyside2", "pyside6",
        "wxpython", "wx", "gtk", "gtk3", "gtk4", "pygobject",
        "kivy", "kivymd", "toga", "dearpygui",
        # Cross-platform
        "cross-platform", "electron", "tauri", "qt",
        # Platform spesifik
        "windows", "windows-app", "macos", "mac-app", "linux-desktop",
        "system-tray", "taskbar", "notification",
        # Oyun
        "game", "game-engine", "pygame", "arcade", "2d", "opengl",
        "graphics", "rendering",
    ],
    "Mobile": [
        "mobile", "mobile-app", "mobile-development",
        "android", "ios", "iphone", "ipad",
        "react-native", "flutter",
        "beeware", "briefcase",
        "kivy", "kivymd",
        "push-notification", "mobile-ui",
    ],
}

print("Kategori keyword sayilari:")
total = 0
for cat, kws in CATEGORY_KEYWORDS.items():
    print(f"  {cat}: {len(kws)} keyword")
    total += len(kws)
print(f"Toplam: {total} keyword")

In [ ]:
def assign_categories(full_name, repo_meta, df_project):
    """
    Bir projeye kategori(ler) ata.
    Kaynak 1: enriched_repos.json'daki topics listesi
    Kaynak 2: description metni
    Kaynak 3: proje adi (full_name)
    En az 1 eslesme: o kategori atanir. Hic eslesme: 'Diger'.
    """
    meta = repo_meta.get(full_name, {})
    topics    = [t.lower() for t in meta.get("topics", [])]
    desc      = (meta.get("description") or "").lower()
    repo_name = full_name.lower()

    # Arama metni: topics + description + proje adi
    search_text = " ".join(topics) + " " + desc + " " + repo_name

    assigned = []
    for category, keywords in CATEGORY_KEYWORDS.items():
        for kw in keywords:
            # Dogru siralama: once escape et, sonra tire'yi [- ] ile degistir
            # Ornek: 'machine-learning'
            #   re.escape  -> 'machine\-learning'
            #   replace    -> 'machine[\- ]learning'
            #   Sonuc: hem 'machine-learning' hem 'machine learning' eslesir
            pattern = r'\b' + re.escape(kw).replace(r'\-', r'[\- ]') + r'\b'
            if re.search(pattern, search_text):
                assigned.append(category)
                break  # Bu kategoriden 1 eslesme yeterli

    return assigned if assigned else ["Diger"]


# Proje listesi
projects = df['project'].unique()

# Her projeye kategori ata
project_categories = {}
for proj in projects:
    cats = assign_categories(proj, repo_meta, df[df['project'] == proj])
    project_categories[proj] = cats

# DataFrame'e ekle
df['categories']       = df['project'].map(project_categories)
df['categories_str']   = df['categories'].apply(lambda x: '|'.join(sorted(x)))
df['primary_category'] = df['categories'].apply(lambda x: x[0])

print(f"Kategorizasyon tamamlandi: {len(projects)} proje")
print(f"  Diger kategorisi: "
      f"{sum(1 for c in project_categories.values() if c == ['Diger'])} proje")

In [ ]:
# Proje bazinda kategori dagilimi
proj_cat = df.drop_duplicates('project')[['project', 'categories_str', 'primary_category']]

print("--- Birincil Kategori Dagilimi (Proje) ---")
primary_counts = proj_cat['primary_category'].value_counts()
for cat, n in primary_counts.items():
    print(f"  {cat}: {n} proje ({n/len(proj_cat)*100:.1f}%)")

print("\n--- Cok Kategorili Projeler ---")
multi = proj_cat[proj_cat['categories_str'].str.contains('|', regex=False)]
print(f"  {len(multi)} proje birden fazla kategoriye giriyor")

# Tum kategorilerde kac proje var (cakisma dahil)
print("\n--- Tum Kategoriler (cakisma dahil) ---")
all_cats = []
for cats in project_categories.values():
    all_cats.extend(cats)
cat_counts = Counter(all_cats)
for cat, n in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {n} proje")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Sol: birincil kategori (proje sayisi)
primary_counts.sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Birincil Kategori — Proje Sayisi', fontweight='bold')
axes[0].set_xlabel('Proje Sayisi')
for i, v in enumerate(primary_counts.sort_values()):
    axes[0].text(v + 0.3, i, str(v), va='center')

# Sag: birincil kategori (dosya sayisi)
file_cat_counts = df['primary_category'].value_counts().sort_values()
file_cat_counts.plot(
    kind='barh', ax=axes[1], color='darkorange', edgecolor='white'
)
axes[1].set_title('Birincil Kategori — Dosya Sayisi', fontweight='bold')
axes[1].set_xlabel('Dosya Sayisi')
for i, v in enumerate(file_cat_counts):
    axes[1].text(v + 5, i, str(v), va='center')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik kaydedildi: category_distribution.png')

---
## Bölüm 3: Filtreleme

Tüm değerler **placeholder** — sonradan değiştirin.

**Proje seviyesi:** star, contributor sayısı, proje yaşı  
**Dosya seviyesi:** commit sayısı, yazar sayısı, dosya yaşı  
**Proje başına dosya:** min/max dosya sayısı  
**Kategori filtresi:** belirli kategorileri seç veya `None` = hepsi

In [ ]:
# ============================================================
# FİLTRE PARAMETRELERİ — placeholder değerler, sonra değiştirin
# ============================================================

# --- Proje seviyesi ---
MIN_STARS          = 0      # placeholder
MAX_STARS          = 10000000  # placeholder
MIN_CONTRIBUTORS   = 1       # placeholder
MAX_CONTRIBUTORS   = 1000000     # placeholder
MIN_PROJECT_AGE    = 7      # placeholder (gun)
MAX_PROJECT_AGE    = 1000000    # placeholder (gun)

# --- Dosya seviyesi ---
MIN_COMMITS        = 1       # placeholder
MAX_COMMITS        = 100000    # placeholder
MIN_AUTHORS        = 1       # placeholder
MIN_FILE_AGE       = 0       # placeholder (gun)

# --- Proje basina dosya ---
MIN_FILES_PER_PROJECT = 3    # placeholder
MAX_FILES_PER_PROJECT = 2000  # placeholder

# --- Kategori filtresi ---
# None = tum kategoriler, liste = sadece secilen kategoriler
SELECTED_CATEGORIES = None   # placeholder
# SELECTED_CATEGORIES = None               # hepsini almak icin bunu kullan
# SELECTED_CATEGORIES = ["AI/ML", "Web"] örnek kategori kullanımı
print("Filtre parametreleri yuklendi.")
print(f"  Proje: stars={MIN_STARS}-{MAX_STARS}, "
      f"contributors={MIN_CONTRIBUTORS}-{MAX_CONTRIBUTORS}, "
      f"age={MIN_PROJECT_AGE}-{MAX_PROJECT_AGE} gun")
print(f"  Dosya: commits={MIN_COMMITS}-{MAX_COMMITS}, "
      f"authors>={MIN_AUTHORS}, file_age>={MIN_FILE_AGE} gun")
print(f"  Proje basina: {MIN_FILES_PER_PROJECT}-{MAX_FILES_PER_PROJECT} dosya")
print(f"  Kategoriler: {SELECTED_CATEGORIES if SELECTED_CATEGORIES else 'Hepsi'}")

In [ ]:
df_filtered = df.copy()
print(f"Baslangic: {len(df_filtered)} dosya, {df_filtered['project'].nunique()} proje")

# -- Proje seviyesi filtreler --
# Her proje icin proje metriklerini al (dosyalar ayni degere sahip)
proj_metrics = df_filtered.drop_duplicates('project').set_index('project')

valid_projects = proj_metrics[
    (proj_metrics['stars'] >= MIN_STARS) &
    (proj_metrics['stars'] <= MAX_STARS) &
    (proj_metrics['contributor_count'] >= MIN_CONTRIBUTORS) &
    (proj_metrics['contributor_count'] <= MAX_CONTRIBUTORS) &
    (proj_metrics['project_age_days'] >= MIN_PROJECT_AGE) &
    (proj_metrics['project_age_days'] <= MAX_PROJECT_AGE)
].index

before = len(df_filtered)
df_filtered = df_filtered[df_filtered['project'].isin(valid_projects)]
print(f"Proje filtresi sonrasi: {len(df_filtered)} dosya, "
      f"{df_filtered['project'].nunique()} proje "
      f"({before - len(df_filtered)} dosya elendi)")

In [ ]:
# -- Dosya seviyesi filtreler --
before = len(df_filtered)

df_filtered = df_filtered[
    (df_filtered['commit_count'] >= MIN_COMMITS) &
    (df_filtered['commit_count'] <= MAX_COMMITS) &
    (df_filtered['n_authors']    >= MIN_AUTHORS) &
    (df_filtered['file_age_days']>= MIN_FILE_AGE)
]

print(f"Dosya filtresi sonrasi: {len(df_filtered)} dosya, "
      f"{df_filtered['project'].nunique()} proje "
      f"({before - len(df_filtered)} dosya elendi)")

In [ ]:
# -- Proje basina dosya filtresi --

# Min filtresi: az dosyali projeleri tamamen at
proj_sizes = df_filtered.groupby('project').size()
keep_projects = proj_sizes[proj_sizes >= MIN_FILES_PER_PROJECT].index
before = len(df_filtered)
df_filtered = df_filtered[df_filtered['project'].isin(keep_projects)]
print(f"Min dosya filtresi: {before - len(df_filtered)} dosya elendi "
      f"({len(proj_sizes) - len(keep_projects)} proje dusuruldu)")

# Max filtresi: buyuk projelerden ornekle
before = len(df_filtered)
df_filtered = (
    df_filtered
    .groupby('project', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), MAX_FILES_PER_PROJECT), random_state=42))
    .reset_index(drop=True)
)
print(f"Max dosya filtresi: {before - len(df_filtered)} dosya orneklendi")
print(f"Sonuc: {len(df_filtered)} dosya, {df_filtered['project'].nunique()} proje")

In [ ]:
# -- Kategori filtresi --
if SELECTED_CATEGORIES is not None:
    before = len(df_filtered)
    # Birincil kategorisi SELECTED_CATEGORIES'de olan projeleri sec
    # Alternatif: herhangi bir kategorisi icinde olsun istiyorsan
    #   df_filtered['categories'].apply(lambda cats: bool(set(cats) & set(SELECTED_CATEGORIES)))
    mask = df_filtered['primary_category'].isin(SELECTED_CATEGORIES)
    df_filtered = df_filtered[mask]
    print(f"Kategori filtresi ({SELECTED_CATEGORIES}): "
          f"{before - len(df_filtered)} dosya elendi")
    print(f"Sonuc: {len(df_filtered)} dosya, {df_filtered['project'].nunique()} proje")
else:
    print("Kategori filtresi uygulanmadi (SELECTED_CATEGORIES=None)")

---
## Bölüm 4: Filtreleme Sonrası Özet

In [ ]:
print('=' * 60)
print('FİLTRELEME SONRASI ÖZET')
print('=' * 60)
print(f"Dosya sayisi  : {len(df_filtered)}")
print(f"Proje sayisi  : {df_filtered['project'].nunique()}")

print("\n--- Sinif Dagilimi (label: commit) ---")
lc = df_filtered['label'].value_counts().sort_index()
for v, n in lc.items():
    print(f"  Sinif {v}: {n} ({n/len(df_filtered):.1%})")

print("\n--- Bug Dagilimi ---")
bc = df_filtered['has_bug'].value_counts().sort_index()
for v, n in bc.items():
    lbl = 'Bug yok' if v == 0 else 'Bug var'
    print(f"  {lbl}: {n} ({n/len(df_filtered):.1%})")

print("\n--- Kategori Dagilimi ---")
cat_dist = df_filtered['primary_category'].value_counts()
for cat, n in cat_dist.items():
    print(f"  {cat}: {n} dosya ({n/len(df_filtered):.1%})")

print("\n--- Proje Agirlik Analizi ---")
pp = df_filtered.groupby('project').size().sort_values(ascending=False)
total = pp.sum()
print(f"  En buyuk proje: {pp.index[0]} ({pp.iloc[0]} dosya, {pp.iloc[0]/total*100:.1f}%)")
print(f"  Ilk 5 proje: {pp.head(5).sum()} dosya ({pp.head(5).sum()/total*100:.1f}%)")
from numpy import cumsum as npcumsum
import numpy as np
cs = np.cumsum(pp.values)
for threshold in [25, 50, 75]:
    n_proj = int(np.searchsorted(cs, total * threshold / 100)) + 1
    print(f"  Verinin %{threshold}'i ilk {n_proj} projeden geliyor")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Label dagilimi
lc_plot = df_filtered['label'].value_counts().sort_index()
axes[0,0].bar(['Median Alti (0)', 'Median Ustu (1)'], lc_plot.values,
              color=['#e74c3c', '#2ecc71'], edgecolor='white')
for i, v in enumerate(lc_plot.values):
    axes[0,0].text(i, v + lc_plot.max()*0.02, f"{v}\n({v/lc_plot.sum():.1%})",
                   ha='center', fontweight='bold')
axes[0,0].set_title('Commit Sinif Dagilimi', fontweight='bold')
axes[0,0].set_ylim(0, lc_plot.max() * 1.25)

# 2. Bug dagilimi
bc_plot = df_filtered['has_bug'].value_counts().sort_index()
axes[0,1].bar(['Bug Yok (0)', 'Bug Var (1)'], bc_plot.values,
              color=['#3498db', '#e67e22'], edgecolor='white')
for i, v in enumerate(bc_plot.values):
    axes[0,1].text(i, v + bc_plot.max()*0.02, f"{v}\n({v/bc_plot.sum():.1%})",
                   ha='center', fontweight='bold')
axes[0,1].set_title('Bug Dagilimi', fontweight='bold')
axes[0,1].set_ylim(0, bc_plot.max() * 1.25)

# 3. Kategori dagilimi
cat_plot = df_filtered['primary_category'].value_counts().sort_values()
cat_plot.plot(kind='barh', ax=axes[1,0], color='#9b59b6', edgecolor='white')
axes[1,0].set_title('Kategori Dagilimi (Dosya)', fontweight='bold')
axes[1,0].set_xlabel('Dosya Sayisi')

# 4. Proje basina dosya
pp_vals = df_filtered.groupby('project').size()
axes[1,1].hist(pp_vals.values, bins=30, color='#1abc9c', edgecolor='white')
axes[1,1].axvline(pp_vals.mean(), color='red', linestyle='--',
                  label=f'Ort: {pp_vals.mean():.0f}')
axes[1,1].axvline(pp_vals.median(), color='orange', linestyle='--',
                  label=f'Med: {pp_vals.median():.0f}')
axes[1,1].set_title('Proje Basina Dosya', fontweight='bold')
axes[1,1].legend()

plt.suptitle(
    f"Filtreleme Sonrasi Dataset ({len(df_filtered)} dosya, "
    f"{df_filtered['project'].nunique()} proje)",
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'filtered_dataset_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik kaydedildi: filtered_dataset_summary.png')

---
## Bölüm 5: Kaydet

In [ ]:
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# 1. Full dataset (code_tokens dahil, her sey)
full_path = OUTPUT_DIR / f"dataset_full_filtered_{timestamp}.csv"
df_filtered.to_csv(full_path, index=False)
print(f"Full dataset: {full_path.name}")
print(f"  {len(df_filtered)} satir, {len(df_filtered.columns)} sutun")

# 2. Model dataset (code_tokens haric, kategoriler dahil)
model_cols = (
    ["project", "file_path"]
    + ["stars", "contributor_count", "project_age_days"]
    + ["commit_count", "bug_count", "n_authors", "file_age_days", "churn_total",
       "avg_churn_per_commit", "max_single_churn", "recent_commits_90d"]
    + ["loc", "lloc", "sloc", "comments", "multi", "blank", "single_comments",
       "cc_mean", "cc_max", "cc_total", "num_functions",
       "h_vocabulary", "h_length", "h_volume", "h_difficulty",
       "h_effort", "h_bugs", "h_time", "h_calculated_length",
       "maintainability_index", "comment_ratio", "doc_ratio"]
    + ["complexity_density", "comment_per_function", "avg_function_length", "effort_per_line"]
    + ["primary_category", "categories_str"]
    + ["label", "has_bug", "bug_severity"]
)

model_path = OUTPUT_DIR / f"dataset_model_filtered_{timestamp}.csv"
df_filtered[model_cols].to_csv(model_path, index=False)
print(f"Model dataset: {model_path.name}")
print(f"  {len(df_filtered)} satir, {len(model_cols)} sutun")

# 3. Kategori-proje mapping'i (referans icin)
cat_map_path = OUTPUT_DIR / f"project_categories_{timestamp}.json"
with open(cat_map_path, 'w', encoding='utf-8') as f:
    json.dump(
        {k: v for k, v in project_categories.items()
         if k in df_filtered['project'].values},
        f, indent=2, ensure_ascii=False
    )
print(f"Kategori map: {cat_map_path.name}")

print(f"\nProje sayisi: {df_filtered['project'].nunique()}")
print(f"\nKaydedilen dosyalar:")
print(f"  1. {full_path.name} — her sey (code_tokens dahil)")
print(f"  2. {model_path.name} — model egitimi icin (kod yok, kategoriler var)")
print(f"  3. {cat_map_path.name} — proje-kategori eslesmesi")
